# 01 - Dataset Split

This notebook creates a fixed training, validation, and test split from the annotated dataset.

The same split will be used for both YOLO and Faster R-CNN so the models are evaluated on the same images.

Split:
- 70% training
- 20% validation
- 10% test

The original annotated dataset is not changed.

In [11]:
from pathlib import Path
import shutil

import pandas as pd
from sklearn.model_selection import train_test_split

# Find the project root whether Jupyter is opened from the project
# folder or from the notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "dataset_annotated_bootstrap").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Existing annotated dataset.
SOURCE_ROOT = PROJECT_ROOT / "dataset_annotated_bootstrap"

# Generated dataset containing the fixed train, validation, and test split.
SPLIT_ROOT = PROJECT_ROOT / "dataset_split"

# CSV file that records which split every image belongs to.
SPLIT_CSV = SPLIT_ROOT / "dataset_split.csv"

# Fixed seed makes the split reproducible.
RANDOM_SEED = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Source dataset: {SOURCE_ROOT}")

Project root: /home/john/dobot-thesis
Source dataset: /home/john/dobot-thesis/dataset_annotated_bootstrap


## Find the annotated images

Each dataset image should have a matching `.txt` annotation file.

This step finds all images and checks that each one has its annotation.

In [12]:
# Image formats used by the dataset.
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
}

records = []
missing_labels = []

# Search through every file in the annotated dataset.
for image_path in sorted(SOURCE_ROOT.rglob("*")):

    if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue

    # The annotation should have the same name as the image but use .txt.
    label_path = image_path.with_suffix(".txt")

    if not label_path.exists():
        missing_labels.append(image_path)
        continue

    # Keep the path relative to the dataset root.
    relative_path = image_path.relative_to(SOURCE_ROOT)

    # Use the dataset folder structure to identify the scene type.
    #
    # Examples:
    # 01_single_blocks/blue_cube
    # 02_two_blocks/red_blue
    # 03_three_blocks/red_blue_green
    parent_parts = relative_path.parent.parts

    if len(parent_parts) >= 2:
        scene_group = f"{parent_parts[0]}/{parent_parts[1]}"
    else:
        scene_group = parent_parts[0]

    records.append(
        {
            "image": relative_path.as_posix(),
            "label": relative_path.with_suffix(".txt").as_posix(),
            "scene_group": scene_group,
        }
    )

dataset_df = pd.DataFrame(records)

print(f"Images found: {len(dataset_df)}")
print(f"Images missing labels: {len(missing_labels)}")

dataset_df.head()

Images found: 370
Images missing labels: 0


,image,label,scene_group
0,01_single_blocks/blue_cube/single_blue_corner_...,01_single_blocks/blue_cube/single_blue_corner_...,01_single_blocks/blue_cube
1,01_single_blocks/blue_cube/single_blue_corner_...,01_single_blocks/blue_cube/single_blue_corner_...,01_single_blocks/blue_cube
2,01_single_blocks/blue_cube/single_blue_corner_...,01_single_blocks/blue_cube/single_blue_corner_...,01_single_blocks/blue_cube
3,01_single_blocks/blue_cube/single_blue_corner_...,01_single_blocks/blue_cube/single_blue_corner_...,01_single_blocks/blue_cube
4,01_single_blocks/blue_cube/single_blue_corner_...,01_single_blocks/blue_cube/single_blue_corner_...,01_single_blocks/blue_cube


## Check the dataset groups

The dataset contains different scenes such as single blocks, two-block colour combinations, three-block combinations, four blocks, and empty workspaces.

The split will use these groups so each part of the dataset contains a similar mixture of scenes.

In [13]:
# Count how many images belong to each scene group.
scene_counts = (
    dataset_df["scene_group"]
    .value_counts()
    .sort_index()
)

scene_counts

scene_group
01_single_blocks/blue_cube              50
01_single_blocks/green_cube             50
01_single_blocks/red_cube               50
01_single_blocks/yellow_cube            50
02_two_blocks/blue_green                15
02_two_blocks/blue_yellow               15
02_two_blocks/red_blue                  15
02_two_blocks/red_green                 15
02_two_blocks/red_yellow                15
02_two_blocks/yellow_green              15
03_three_blocks/blue_yellow_green       10
03_three_blocks/red_blue_green          10
03_three_blocks/red_blue_yellow         10
03_three_blocks/red_yellow_green        10
04_four_blocks/red_blue_yellow_green    20
05_empty_workspace                      20
Name: count, dtype: int64

## Create the train, validation, and test split

The dataset is split into:

- 70% training
- 20% validation
- 10% test

The split is stratified by scene group. This helps keep the different block combinations represented across all three sets.

A fixed random seed is used so running the notebook again produces the same split.

In [14]:
# First keep 70% for training and 30% for validation + testing.
train_df, remaining_df = train_test_split(
    dataset_df,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=dataset_df["scene_group"],
)

# Split the remaining 30% into:
#
# 20% validation
# 10% test
#
# One third of the remaining 30% becomes the test set.
val_df, test_df = train_test_split(
    remaining_df,
    test_size=1 / 3,
    random_state=RANDOM_SEED,
    stratify=remaining_df["scene_group"],
)

# Record which split each image belongs to.
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

# Put all three parts back into one table.
split_df = pd.concat(
    [
        train_df,
        val_df,
        test_df,
    ],
    ignore_index=True,
)

# Sort the table so the saved CSV is easy to read.
split_df = split_df.sort_values(
    [
        "split",
        "scene_group",
        "image",
    ]
).reset_index(drop=True)

## Check the split

This step confirms how many images were assigned to training, validation, and testing.

In [15]:
# Count the number of images in each split.
split_counts = split_df["split"].value_counts()

for split_name in [
    "train",
    "val",
    "test",
]:
    count = split_counts[split_name]
    percentage = count / len(split_df) * 100

    print(
        f"{split_name:5}: "
        f"{count:3} images "
        f"({percentage:.1f}%)"
    )

train: 259 images (70.0%)
val  :  74 images (20.0%)
test :  37 images (10.0%)


## Check the scene distribution

This table shows how each type of scene was divided between training, validation, and testing.

It is useful for checking that one scene type has not accidentally been placed entirely in one split.

In [16]:
# Show how each scene group is distributed across the three splits.
scene_split_table = pd.crosstab(
    split_df["scene_group"],
    split_df["split"],
)

scene_split_table

split,test,train,val
scene_group,,,
01_single_blocks/blue_cube,5,35,10
01_single_blocks/green_cube,5,35,10
01_single_blocks/red_cube,5,35,10
01_single_blocks/yellow_cube,5,35,10
02_two_blocks/blue_green,1,11,3
02_two_blocks/blue_yellow,1,11,3
02_two_blocks/red_blue,2,10,3
02_two_blocks/red_green,2,10,3
02_two_blocks/red_yellow,2,10,3


## Create the shared split dataset

The original annotated dataset will remain unchanged.

A new `dataset_split` folder is created with separate image and label folders for training, validation, and testing.

The original nested folder structure is kept so filenames cannot clash.

In [17]:
# Delete the previous generated split if this notebook has already been run.
if SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)

# Create the new split folder.
SPLIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# Copy every image and annotation into its assigned split.
for row in split_df.itertuples(index=False):
    relative_image = Path(row.image)
    relative_label = Path(row.label)

    source_image = SOURCE_ROOT / relative_image
    source_label = SOURCE_ROOT / relative_label

    target_image = (
            SPLIT_ROOT
            / "images"
            / row.split
            / relative_image
    )

    target_label = (
            SPLIT_ROOT
            / "labels"
            / row.split
            / relative_label
    )

    # Create any required nested folders.
    target_image.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    target_label.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Copy the image and its matching annotation.
    shutil.copy2(
        source_image,
        target_image,
    )

    shutil.copy2(
        source_label,
        target_label,
    )

print("Dataset split created.")

Dataset split created.


## Save the split

The split is also saved to a CSV file.

This file is important because the Faster R-CNN experiment will use the same train, validation, and test assignments as YOLO.

In [18]:
# Save the exact split so it can be reused by both models.
split_df.to_csv(
    SPLIT_CSV,
    index=False,
)

print(f"Split saved to: {SPLIT_CSV}")

Split saved to: /home/john/dobot-thesis/dataset_split/dataset_split.csv


## Final check

Confirm that the generated dataset contains the same number of images and annotations as the source dataset.

In [19]:
for split_name in [
    "train",
    "val",
    "test",
]:
    image_count = sum(
        1
        for path in (
                SPLIT_ROOT
                / "images"
                / split_name
        ).rglob("*")
        if path.suffix.lower() in IMAGE_EXTENSIONS
    )

    label_count = len(
        list(
            (
                    SPLIT_ROOT
                    / "labels"
                    / split_name
            ).rglob("*.txt")
        )
    )

    print(
        f"{split_name:5}: "
        f"{image_count} images, "
        f"{label_count} labels"
    )

total_split_images = sum(
    1
    for path in (SPLIT_ROOT / "images").rglob("*")
    if path.suffix.lower() in IMAGE_EXTENSIONS
)

print()
print(f"Original images: {len(dataset_df)}")
print(f"Split images:    {total_split_images}")

train: 259 images, 259 labels
val  : 74 images, 74 labels
test : 37 images, 37 labels

Original images: 370
Split images:    370
